# Phase 2 · Proposed variants and ablations on CIFAR-100

Start only after completing and downloading **Phase 1**.
Attach **alincijov/cifar-100** plus the saved Phase 1 notebook Output, or a Kaggle Dataset
containing its `result.zip` / extracted `result/`. Set `PHASE1_SOURCE` when needed.

This notebook restores **the exact four evaluated Phase 1 models**, checks SHA-256 hashes,
and verifies the dataset, class mapping and normalization. Original/Retrain/Shrink/Expanding
are fixed references: there are no baseline training cells and no fallback training.

Run the three proposed variants independently: distance weights, Original KL, then Shrink KL
(warm start from distance weights, so this last student has a larger update budget).
Default epoch/LR/batch size come from Phase 1; BN update stays the default.
Optional BN and weighting ablations are separate, disabled cells at the end.
Outputs and copied Phase 1 references are included in **result.zip**.


In [ ]:
# Run this cell first, both locally and on Kaggle.
from pathlib import Path
import importlib
import subprocess
import sys

GITHUB_REPO = "https://github.com/ngvihoa/boundary-unlearning.git"  
GITHUB_REF = "main"  # Branch, tag, or commit SHA. Use a SHA to repeat an experiment.
REPO_DIR = Path("/kaggle/working/boundary-unlearning")

if GITHUB_REPO:
    if not (REPO_DIR / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--no-checkout",
            GITHUB_REPO, str(REPO_DIR),
        ], check=True)
    subprocess.run([
        "git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", GITHUB_REF,
    ], check=True)
    subprocess.run([
        "git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD",
    ], check=True)
    CODE_ROOT = REPO_DIR
else:
    CODE_ROOT = Path.cwd()

if not (CODE_ROOT / "boundary_unlearning" / "__init__.py").is_file():
    raise FileNotFoundError(
        "Set GITHUB_REPO to your GitHub HTTPS URL on Kaggle, "
        "or run this notebook from the local repository root."
    )

sys.path.insert(0, str(CODE_ROOT))
importlib.invalidate_caches()
print("Code root:", CODE_ROOT)
if (CODE_ROOT / ".git").exists():
    print("Code commit:", subprocess.check_output(
        ["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"], text=True,
    ).strip())
# For small code changes, reload the affected module and rerun its caller cells only.


In [ ]:
import csv
import hashlib
import json
import platform
import random
from pathlib import Path
from functools import partial

import pandas as pd
import numpy as np
import torch
import torchvision
from torch.utils.data import Subset
from IPython.display import display
from boundary_unlearning import cifar100, data, models, training, unlearning, phases

# Attach the CIFAR-100 dataset AND the completed Phase 1 Output/result.zip.
# None auto-selects one attached Phase 1 bundle; set a path if multiple runs are attached.
PHASE1_SOURCE = None  # e.g. Path("/kaggle/input/my-phase1-results/result.zip")
INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = Path("/kaggle/input/datasets/alincijov/cifar-100")
if not DATA_ROOT.is_dir() and Path("/kaggle/input/cifar-100").is_dir():
    DATA_ROOT = Path("/kaggle/input/cifar-100")
OUTPUT_ROOT = Path("/kaggle/working")
RESULT_ROOT = OUTPUT_ROOT / "result"
CHECKPOINT_ROOT = RESULT_ROOT / "checkpoints"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load validated evaluated snapshots. No train_from_scratch/create_or_load calls.
phase1_source = phases.find_phase1_source(INPUT_ROOT) if PHASE1_SOURCE is None else Path(PHASE1_SOURCE)
phase1 = phases.load_phase1_bundle(phase1_source, device=DEVICE, copy_to=RESULT_ROOT / "phase1_source")
phase1_config = phase1["config"]
phase1_rows = phase1["rows"]
EXPERIMENT_PHASE = 2
DATASET_NAME = phase1_config["dataset"]
DATASET_SLUG = phase1_config["dataset_slug"]
LABEL_TYPE = phase1_config["label_type"]
NUM_CLASSES = phase1_config["num_classes"]
FORGET_CLASS = phase1_config["forget_class"]
SEED = phase1_config["seed"]
BATCH_SIZE = phase1_config["batch_size"]
NUM_WORKERS = phase1_config["num_workers"]
RUN_PROFILE = phase1_config["run_profile"]
ORIGINAL_EPOCHS = phase1_config["original_epochs"]
ORIGINAL_LR = phase1_config["original_learning_rate"]
UNLEARN_EPOCHS = phase1_config["unlearn_epochs"]
UNLEARN_LR = phase1_config["unlearn_learning_rate"]
MOMENTUM = phase1_config["momentum"]
FGSM_BOUND = phase1_config["fgsm_bound"]
MAX_BATCHES_PER_EPOCH = phase1_config["max_batches_per_epoch"]
EVALUATE_TRAIN_SPLITS = phase1_config["evaluate_train_splits"]
ATTACK_NORMALIZATION = phase1_config["attack_normalization"]
CIFAR_MEAN = tuple(phase1_config["input_mean"])
CIFAR_STD = tuple(phase1_config["input_std"])
REPO_ATTACK_MEAN = tuple(phase1_config["attack_mean"])
REPO_ATTACK_STD = tuple(phase1_config["attack_std"])
EXPECTED_TRAIN_IMAGES = phase1_config["train_sample_count"]
EXPECTED_TEST_IMAGES = phase1_config["test_sample_count"]
EXTRA_EXP = None
BN_STATS_MODE = "update"  # Match Phase 1; use optional paired cells to study BN freeze.

# Proposed variants; keep them separate from the fixed, restored Phase 1 baselines.
WEIGHT_EPSILON_STEP = 1 / 255
RETAIN_KL_LAMBDA = 0.1
RETAIN_KL_TEMPERATURE = 2.0
METHODS_TO_RUN = ["boundary_shrink", "boundary_expanding",
                  unlearning.DISTANCE_METHOD, unlearning.ORIGINAL_KL_METHOD, unlearning.BOUNDARY_KL_METHOD]
original_model, retrain_model = phase1["models"]["original"], phase1["models"]["retrain"]
unlearned_models = {name: phase1["models"][name] for name in ("boundary_shrink", "boundary_expanding")}
unlearn_stats = {name: dict(phase1_config["unlearning_stats"][name]) for name in unlearned_models}
phase1_times = {row["model"]: row.get("training_seconds") for row in phase1_rows}
original_train_seconds, retrain_train_seconds = phase1_times["original"], phase1_times["retrain"]
baseline_tag = phase1_config["baseline_tag"]
baseline_tag_settings = phase1_config["baseline_tag_settings"]
original_checkpoint = phase1["output_dir"] / phase1["manifest"]["models"]["original"]["path"]
retrain_checkpoint = phase1["output_dir"] / phase1["manifest"]["models"]["retrain"]["path"]

tag_settings = {"experiment_phase": 2, "source_phase1_run": phase1["manifest"]["run_tag"],
                "source_phase1_manifest_sha256": phase1["manifest_sha256"],
                "methods": METHODS_TO_RUN, "seed": SEED, "unlearn_epochs": UNLEARN_EPOCHS,
                "unlearn_lr": UNLEARN_LR, "momentum": MOMENTUM, "batch_size": BATCH_SIZE,
                "bn_stats_mode": BN_STATS_MODE, "epsilon_step": WEIGHT_EPSILON_STEP,
                "retain_kl_lambda": RETAIN_KL_LAMBDA, "retain_kl_temperature": RETAIN_KL_TEMPERATURE}
tag_hash = hashlib.sha256(json.dumps(tag_settings, sort_keys=True).encode()).hexdigest()[:8]
run_tag = f"cifar100_phase2_{RUN_PROFILE}_forget{FORGET_CLASS}_{tag_hash}"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("Phase 1 source:", phase1_source)
print("Phase 1 run:", phase1["manifest"]["run_tag"])
print("Phase 2 run:", run_tag)
print("Restored models:", list(phase1["models"]))
display(pd.DataFrame(phase1_rows).set_index("model"))


In [ ]:
# Load the attached Kaggle pickles, including the labeled test split.
DATA_ROOT = cifar100.resolve_data_root(DATA_ROOT)
train_dataset, evaluation_dataset = cifar100.load_cifar100(
    DATA_ROOT, mean=CIFAR_MEAN, std=CIFAR_STD,
)
CIFAR_CLASSES = train_dataset.classes
train_labels = train_dataset.targets
test_labels = evaluation_dataset.targets

if len(train_dataset) != EXPECTED_TRAIN_IMAGES or len(evaluation_dataset) != EXPECTED_TEST_IMAGES:
    raise ValueError("Expected the complete CIFAR-100 train/test splits (50,000 / 10,000)")
train_class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
test_class_counts = np.bincount(test_labels, minlength=NUM_CLASSES)
if not np.all(train_class_counts == EXPECTED_TRAIN_IMAGES // NUM_CLASSES):
    raise ValueError("Training fine-label counts do not match the balanced CIFAR-100 split")
if not np.all(test_class_counts == EXPECTED_TEST_IMAGES // NUM_CLASSES):
    raise ValueError("Test fine-label counts do not match the balanced CIFAR-100 split")

display(pd.DataFrame([
    {"Split": "Train", "Images": len(train_dataset), "Classes": NUM_CLASSES,
     "Images per class": int(train_class_counts[FORGET_CLASS])},
    {"Split": "Test", "Images": len(evaluation_dataset), "Classes": NUM_CLASSES,
     "Images per class": int(test_class_counts[FORGET_CLASS])},
]).set_index("Split"))
print(f"Forget class: {FORGET_CLASS} ({CIFAR_CLASSES[FORGET_CLASS]})")

phases.validate_phase1_dataset(phase1_config, train_dataset, evaluation_dataset)
print("Verified: identical Phase 1 pixels, labels, class mapping and normalization.")


In [ ]:
train_fit_set = train_dataset
train_forget_indices = np.flatnonzero(train_labels == FORGET_CLASS).tolist()
train_retain_indices = np.flatnonzero(train_labels != FORGET_CLASS).tolist()
test_labels = np.asarray(evaluation_dataset.targets, dtype=np.int64)
test_forget_indices = np.flatnonzero(test_labels == FORGET_CLASS).tolist()
test_retain_indices = np.flatnonzero(test_labels != FORGET_CLASS).tolist()

train_forget_set = Subset(train_dataset, train_forget_indices)
train_retain_set = Subset(train_dataset, train_retain_indices)
test_forget_set = Subset(evaluation_dataset, test_forget_indices)
test_retain_set = Subset(evaluation_dataset, test_retain_indices)

print("Train fit:", len(train_fit_set))
print("Train forget:", len(train_forget_set))
print("Train retain:", len(train_retain_set))
print("Official test:", len(evaluation_dataset))
print("Official test forget:", len(test_forget_set))
print("Official test retain:", len(test_retain_set))


In [ ]:
PIN_MEMORY = DEVICE.type == "cuda"
make_loader = partial(
    data.make_loader, batch_size=BATCH_SIZE, seed=SEED,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)

train_all_loader = make_loader(train_fit_set, shuffle=True)
train_retain_loader = make_loader(train_retain_set, shuffle=True)
train_retain_eval_loader = make_loader(train_retain_set, shuffle=False)
train_forget_eval_loader = make_loader(train_forget_set, shuffle=False)

test_loader = make_loader(evaluation_dataset, shuffle=False)
test_forget_loader = make_loader(test_forget_set, shuffle=False)
test_retain_loader = make_loader(test_retain_set, shuffle=False)

class_count = np.bincount(train_labels, minlength=NUM_CLASSES)

print(f"Fine classes: {len(CIFAR_CLASSES)} | Forget samples per epoch: {len(train_forget_set)}")
print("Train all batches:", len(train_all_loader))
print("Train retain batches:", len(train_retain_loader))
print("Train retain evaluation batches:", len(train_retain_eval_loader))
print("Train forget evaluation batches:", len(train_forget_eval_loader))
print("Official test batches:", len(test_loader))
print("Official test forget batches:", len(test_forget_loader))
print("Official test retain batches:", len(test_retain_loader))

forget_batches = len(make_loader(train_forget_set))
if MAX_BATCHES_PER_EPOCH is not None:
    forget_batches = min(forget_batches, MAX_BATCHES_PER_EPOCH)
print("Unlearning steps per method:", UNLEARN_EPOCHS * forget_batches)


## Proposed variants

The four Phase 1 baselines are already loaded. Each cell below trains only one proposed variant.


In [ ]:
# Incremental execution: only new variants train here; Phase 1 models stay fixed.
from boundary_unlearning import attacks, enhanced
for module in (attacks, enhanced, unlearning):
    importlib.reload(module)
unlearning.validate_methods(METHODS_TO_RUN, extra_exp=EXTRA_EXP)
run_boundary_distance = partial(
    unlearning.run_boundary_distance, device=DEVICE, epochs=UNLEARN_EPOCHS,
    learning_rate=UNLEARN_LR, momentum=MOMENTUM, pin_memory=PIN_MEMORY,
    max_batches_per_epoch=MAX_BATCHES_PER_EPOCH, bn_stats_mode=BN_STATS_MODE,
    bound=FGSM_BOUND, epsilon_step=WEIGHT_EPSILON_STEP,
    kl_lambda=RETAIN_KL_LAMBDA, temperature=RETAIN_KL_TEMPERATURE,
    cifar_mean=CIFAR_MEAN, cifar_std=CIFAR_STD,
)

def run_method(method_name):
    if method_name in phases.PHASE1_MODELS:
        raise ValueError("Phase 1 baselines are restored and fixed; do not retrain them in Phase 2")
    if method_name not in METHODS_TO_RUN:
        raise ValueError(f"Method not selected: {method_name}")
    if method_name == unlearning.BOUNDARY_KL_METHOD:
        required = ("boundary_shrink", unlearning.DISTANCE_METHOD)
        missing = [name for name in required if name not in unlearned_models or name not in unlearn_stats]
        if missing:
            raise RuntimeError(f"Run prerequisite method cells first: {missing}")
    model, stats = run_boundary_distance(
        original_model, make_loader(train_forget_set, shuffle=True),
        CHECKPOINT_ROOT / f"{method_name}_{run_tag}.pt", method=method_name,
        retain_loader=make_loader(train_retain_set, shuffle=True),
        boundary_model=unlearned_models.get("boundary_shrink"),
        weight_assign_model=unlearned_models.get(unlearning.DISTANCE_METHOD),
        dependency_stats=unlearn_stats,
    )
    if method_name == unlearning.DISTANCE_METHOD:
        unlearned_models.pop(unlearning.BOUNDARY_KL_METHOD, None)
        unlearn_stats.pop(unlearning.BOUNDARY_KL_METHOD, None)
    unlearned_models[method_name], unlearn_stats[method_name] = model, stats
    print(f"Ready: {method_name}")


In [ ]:
# Distance weights: initialize from Original
run_method("boundary_shrink_weight_assign")


In [ ]:
# Distance + Original KL: initialize from Original
run_method("boundary_shrink_weight_assign_distill")


In [ ]:
# Distance + Shrink KL: warm start from distance weights
run_method("boundary_shrink_weight_assign_boundary_distill")


In [ ]:
from boundary_unlearning.reporting import results_table, unlearning_details_table

evaluate_model_splits = partial(
    training.evaluate_model_splits, device=DEVICE,
    test_loader=test_loader, test_retain_loader=test_retain_loader,
    test_forget_loader=test_forget_loader,
    train_retain_eval_loader=train_retain_eval_loader,
    train_forget_eval_loader=train_forget_eval_loader,
)

# ============================================================
# CIFAR-100 FINE-LABEL TEST EVALUATION
# ============================================================

missing_methods = [name for name in METHODS_TO_RUN if name not in unlearned_models]
if missing_methods:
    raise RuntimeError(f"Run the missing method cells before final evaluation: {missing_methods}")

models_to_evaluate = {
    "original": original_model,
    "retrain": retrain_model,
    **{name: unlearned_models[name] for name in METHODS_TO_RUN},
}

rows = []
baseline_train_times = {"original": original_train_seconds, "retrain": retrain_train_seconds}

for name, model in models_to_evaluate.items():
    row = {"model": name}
    row.update(evaluate_model_splits(model, include_train=EVALUATE_TRAIN_SPLITS))

    if name in baseline_train_times:
        row["training_seconds"] = baseline_train_times[name]
    else:
        row["training_seconds"] = None

    row["unlearning_seconds"] = unlearn_stats[name]["seconds"] if name in unlearn_stats else None
    row["fgsm_cross_rate"] = unlearn_stats[name].get("cross_rate") if name in unlearn_stats else None
    row["unlearning_update_steps"] = unlearn_stats[name].get("update_steps") if name in unlearn_stats else None

    stats = unlearn_stats.get(name, {})
    row["total_unlearning_seconds"] = stats.get("total_seconds", stats.get("seconds"))
    row["student_update_steps"] = stats.get("student_update_steps", stats.get("update_steps"))
    row["total_update_steps"] = stats.get("total_update_steps", stats.get("update_steps"))
    row["mean_sample_weight"] = stats.get("mean_sample_weight")
    row["mean_boundary_epsilon"] = stats.get("mean_boundary_epsilon")
    row["initializer"] = stats.get("initializer", "original" if stats else None)
    row["retain_teacher"] = stats.get("retain_teacher")

    rows.append(row)

display(results_table(rows, dataset_name=DATASET_NAME))
display(unlearning_details_table(rows))

metrics_path = RESULT_ROOT / f"{run_tag}_metrics.csv"
with metrics_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

config = {
    "experiment_phase": EXPERIMENT_PHASE,
    "dataset": DATASET_NAME, "dataset_slug": DATASET_SLUG,
    "dataset_url": f"https://www.kaggle.com/datasets/{DATASET_SLUG}",
    "num_classes": NUM_CLASSES, "label_type": LABEL_TYPE, "class_names": list(CIFAR_CLASSES),
    "model_architecture": {"n_channels": 3, "num_classes": NUM_CLASSES,
                           "filters_percentage": 0.5, "batch_norm": True},
    "input_mean": CIFAR_MEAN, "input_std": CIFAR_STD,
    "attack_mean": CIFAR_MEAN if ATTACK_NORMALIZATION == "matched" else REPO_ATTACK_MEAN,
    "attack_std": CIFAR_STD if ATTACK_NORMALIZATION == "matched" else REPO_ATTACK_STD,
    "run_tag_settings": tag_settings, "baseline_tag": baseline_tag,
    "baseline_tag_settings": baseline_tag_settings,
    "baseline_checkpoints": {"original": str(original_checkpoint), "retrain": str(retrain_checkpoint)},
    "baseline_checkpoint_policy": "restored_exact_phase1_evaluated_snapshots",
    "unlearning_stats": {name: unlearn_stats[name] for name in METHODS_TO_RUN},
    "run_profile": RUN_PROFILE, "data_root": str(DATA_ROOT), "methods": METHODS_TO_RUN,
    "forget_class": FORGET_CLASS, "forget_class_name": CIFAR_CLASSES[FORGET_CLASS],
    "seed": SEED, "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
    "evaluation_dataset": "kaggle_cifar100_test_fine",
    "train_sample_count": len(train_fit_set), "test_sample_count": len(evaluation_dataset),
    "extra_exp": EXTRA_EXP, "bn_stats_mode": BN_STATS_MODE,
    "original_epochs": ORIGINAL_EPOCHS, "original_learning_rate": ORIGINAL_LR,
    "unlearn_epochs": UNLEARN_EPOCHS, "unlearn_learning_rate": UNLEARN_LR,
    "momentum": MOMENTUM, "fgsm_bound": FGSM_BOUND,
    "attack_normalization": ATTACK_NORMALIZATION,
    "max_batches_per_epoch": MAX_BATCHES_PER_EPOCH, "evaluate_train_splits": EVALUATE_TRAIN_SPLITS,
    "train_data_shape": list(train_dataset.data.shape), "test_data_shape": list(evaluation_dataset.data.shape),
    "device": str(DEVICE),
    "code_commit": subprocess.check_output(["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"], text=True).strip()
                   if (Path(CODE_ROOT) / ".git").exists() else None,
    "environment": {"python": platform.python_version(), "torch": torch.__version__,
                    "torchvision": torchvision.__version__, "numpy": np.__version__},
}
config.update(
    data_identity=phase1_config["data_identity"],
    source_phase1_run=phase1["manifest"]["run_tag"],
    source_phase1_manifest_sha256=phase1["manifest_sha256"],
    source_phase1_checkpoint_hashes={name: entry["sha256"] for name, entry in phase1["manifest"]["models"].items()},
    distance_search={"epsilon_step": WEIGHT_EPSILON_STEP, "normalization": "matched"},
    retain_distillation={"lambda": RETAIN_KL_LAMBDA, "temperature": RETAIN_KL_TEMPERATURE},
)
config_path = RESULT_ROOT / f"{run_tag}_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Metrics:", metrics_path)
print("Configuration:", config_path)


In [ ]:
# Small code update: uncomment these two commands to fetch code from GitHub.
# !git -C /kaggle/working/boundary-unlearning fetch --depth 1 origin main
# !git -C /kaggle/working/boundary-unlearning checkout --detach FETCH_HEAD

import importlib
from boundary_unlearning import attacks, enhanced, unlearning, reporting, artifacts, visualization, phases
importlib.invalidate_caches()
for module in (attacks, enhanced, unlearning, reporting, artifacts, visualization, phases):
    importlib.reload(module)
print("For plot-only changes: rerun Visualization setup and affected plot cells, then export.")
print("Modules reloaded. Rerun runner setup, the affected method cell, evaluation, then export.")
print("Existing Original/Retrain models remain in memory; their training cells are unnecessary.")


## Visualize the seven-model comparison

These cells use existing models and evaluation rows; they never train or reload checkpoints.
Run **Visualization setup** after changing models or settings, then any plot cell independently.
The summary plots need only `rows`; Grad-CAM/entropy/t-SNE use the selected model objects.

Sampling is deterministic and shared across models. Grad-CAM pages show the forgotten class
with prediction confidence and P(forget); each heatmap has its own intensity normalization.
Entropy is a descriptive diagnostic. t-SNE panels use separate embeddings with unaligned axes;
CIFAR-100 retain classes share a gray color. Neither plot alone establishes successful unlearning.

Outputs: `result/figures/<run_tag>/` with PNG/PDF, metadata JSON and raw arrays (NPZ).
The final export cell includes every generated figure. t-SNE can be skipped when time is limited.


In [ ]:
# VISUALIZATION SETUP: safe to rerun with existing models; no training.
import importlib
import json
from pathlib import Path
import matplotlib.pyplot as plt
from boundary_unlearning import visualization
importlib.reload(visualization)

VIS_DATASET_NAME = "CIFAR-100" if NUM_CLASSES == 100 else "CIFAR-10"
FIGURE_ROOT = Path(OUTPUT_ROOT) / "result" / "figures" / run_tag
VIS_SEED = SEED
VIS_BATCH_SIZE = 128
VIS_GRADCAM_SAMPLES = 4
VIS_ENTROPY_SAMPLES = 2000  # Per split, bounded before inference.
VIS_TSNE_SAMPLES = 1500    # Total across all classes, including CIFAR-100.
VIS_TSNE_PER_CLASS = 120
VIS_TSNE_PERPLEXITY = 30
VIS_TSNE_ITERATIONS = 1000

missing = [name for name in METHODS_TO_RUN if name not in unlearned_models]
if missing:
    raise RuntimeError(f"Run missing method cells first: {missing}")
# Read model objects again when setup is rerun, avoiding stale model references.
visualization_models = {
    "original": original_model,
    "retrain": retrain_model,
    **{name: unlearned_models[name] for name in METHODS_TO_RUN},
}
visualization_datasets = {
    "train_retain": train_retain_set,
    "train_forget": train_forget_set,
    "test": evaluation_dataset,
}
gradcam_indices = visualization.select_sample_indices(
    train_forget_set, max_samples=VIS_GRADCAM_SAMPLES, seed=VIS_SEED,
)
entropy_indices = {
    split: visualization.select_sample_indices(dataset, max_samples=VIS_ENTROPY_SAMPLES, seed=VIS_SEED)
    for split, dataset in visualization_datasets.items()
}
tsne_indices = visualization.select_sample_indices(
    evaluation_dataset, max_samples=VIS_TSNE_SAMPLES, per_class=VIS_TSNE_PER_CLASS, seed=VIS_SEED,
)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
visualization_settings = {
    "run_tag": run_tag, "dataset": VIS_DATASET_NAME, "seed": VIS_SEED,
    "models": list(visualization_models), "class_names": list(CIFAR_CLASSES),
    "forget_class": FORGET_CLASS, "input_mean": CIFAR_MEAN, "input_std": CIFAR_STD,
    "gradcam_split": "train_forget", "gradcam_indices": gradcam_indices.tolist(),
    "entropy_indices": {split: indices.tolist() for split, indices in entropy_indices.items()},
    "tsne_split": "test", "tsne_indices": tsne_indices.tolist(),
    "tsne_perplexity": VIS_TSNE_PERPLEXITY, "tsne_max_iter": VIS_TSNE_ITERATIONS,
    "entropy_sample_limit": VIS_ENTROPY_SAMPLES, "tsne_sample_limit": VIS_TSNE_SAMPLES,
    "tsne_per_class_limit": VIS_TSNE_PER_CLASS, "inference_batch_size": VIS_BATCH_SIZE,
    "model_source": "current_in_memory_models", "data_root": str(DATA_ROOT),
}
(FIGURE_ROOT / "visualization_config.json").write_text(
    json.dumps(visualization_settings, ensure_ascii=False, indent=2), encoding="utf-8",
)
print("Figure folder:", FIGURE_ROOT)
print("Models:", len(visualization_models), "| t-SNE samples:", len(tsne_indices))


In [ ]:
# VISUALIZATION: accuracy and cost. Can run without model objects or visualization setup.
from pathlib import Path
import importlib
import matplotlib.pyplot as plt
from boundary_unlearning import visualization
importlib.reload(visualization)
FIGURE_ROOT = Path(OUTPUT_ROOT) / "result" / "figures" / run_tag
VIS_DATASET_NAME = "CIFAR-100" if NUM_CLASSES == 100 else "CIFAR-10"
fig = visualization.plot_metric_comparison(rows, output_dir=FIGURE_ROOT, dataset_name=VIS_DATASET_NAME)
plt.show()
plt.close(fig)
fig = visualization.plot_cost_comparison(rows, output_dir=FIGURE_ROOT, dataset_name=VIS_DATASET_NAME)
plt.show()
plt.close(fig)


In [ ]:
# VISUALIZATION: Grad-CAM on fixed forget samples; no weight updates.
for fig in visualization.plot_gradcam_comparison(
    visualization_models, train_forget_set, gradcam_indices,
    device=DEVICE, class_names=CIFAR_CLASSES, forget_class=FORGET_CLASS,
    mean=CIFAR_MEAN, std=CIFAR_STD, output_dir=FIGURE_ROOT,
    dataset_name=VIS_DATASET_NAME, models_per_page=4,
):
    plt.show()
    plt.close(fig)


In [ ]:
# VISUALIZATION: entropy distributions on the same selected samples per model.
fig = visualization.plot_entropy_comparison(
    visualization_models, visualization_datasets, entropy_indices,
    device=DEVICE, batch_size=VIS_BATCH_SIZE,
    output_dir=FIGURE_ROOT, dataset_name=VIS_DATASET_NAME,
)
plt.show()
plt.close(fig)


In [ ]:
# VISUALIZATION: optional t-SNE; skip this cell for a quick export.
fig = visualization.plot_tsne_comparison(
    visualization_models, evaluation_dataset, tsne_indices,
    device=DEVICE, class_names=CIFAR_CLASSES, forget_class=FORGET_CLASS,
    seed=VIS_SEED, perplexity=VIS_TSNE_PERPLEXITY, max_iter=VIS_TSNE_ITERATIONS,
    batch_size=VIS_BATCH_SIZE, output_dir=FIGURE_ROOT, dataset_name=VIS_DATASET_NAME,
)
plt.show()
plt.close(fig)


## Optional: paired BatchNorm ablation (existing models only)

After updating code, set `RUN_BN_ABLATION = True` below and run this one cell.
It runs plain Boundary Shrink twice from the same in-memory Original: BN statistics
**update** and **freeze**. Epoch/LR/batch size stay unchanged. Original and Retrain
are only evaluated. Existing seven-model results are preserved.

Outputs go to `result/bn_ablation/<source_run_tag>_<hash>/`, with checkpoints,
CSV/JSON/HTML, loss curves and a separate Original-teacher epsilon/weight audit.
The cell exports `result.zip` directly; no need to rerun main evaluation or visualization.
To investigate distance/KL later, extend `BN_ABLATION_METHODS` with the supported names.


In [ ]:
# BN ABLATION: enable and run only this cell in the existing trained session.
RUN_BN_ABLATION = False
BN_ABLATION_METHODS = ["boundary_shrink"]
# Optional later: "boundary_shrink_weight_assign", "boundary_shrink_weight_assign_distill".

if RUN_BN_ABLATION:
    from pathlib import Path
    bn_script = Path(CODE_ROOT) / "experiments" / "run-bn-ablation.py"
    exec(compile(bn_script.read_text(), str(bn_script), "exec"), globals())
else:
    print("BN ablation skipped. Set RUN_BN_ABLATION=True to run only this diagnostic cell.")


## Optional: distance weights versus weights = 1

Paired frozen-BN control, without KL. Run only when investigating the contribution of weighting.


In [ ]:
# WEIGHTING ABLATION: opt in separately; does not replace main-flow results.
RUN_WEIGHTING_ABLATION = False
if RUN_WEIGHTING_ABLATION:
    exec((Path(CODE_ROOT) / "experiments" / "run-weighting-ablation.py").read_text())
else:
    print("Weighting ablation skipped.")


In [ ]:
# ============================================================
# EXPORT RESULTS: run this cell independently to refresh the ZIP.
# ============================================================
from pathlib import Path
import os
from IPython.display import FileLink, display
from boundary_unlearning.artifacts import export_results
from boundary_unlearning.reporting import results_table, unlearning_details_table

# Use the singular folder name even in a session using the older path settings.
result_dir = Path(OUTPUT_ROOT) / "result"
result_dir.mkdir(parents=True, exist_ok=True)
table_path = result_dir / f"{run_tag}_results.html"
results_table(rows, dataset_name=DATASET_NAME).to_html(table_path)

details_path = result_dir / f"{run_tag}_unlearning_details.html"
unlearning_details_table(rows).to_html(details_path)

# Includes existing result/figures/ recursively; no plot is recomputed here.
archive_path = export_results(
    result_dir,
    checkpoint_dir=CHECKPOINT_ROOT,
    files=[metrics_path, config_path],
)
print(f"Archive: {archive_path} ({archive_path.stat().st_size / (1024 ** 2):.2f} MiB)")
display(FileLink(os.path.relpath(archive_path), result_html_prefix="Download: "))
